# Implementación: Softmax Causal y Paralelización Matricial

Vamos a llevar a la práctica la **transformación de probabilidad Softmax ignorando la información del futuro**.

---

## 1. Repaso de la Intuición

Supongamos que tenemos una secuencia de palabras procesadas como tokens:

$$\text{Secuencia} = [\text{"El"}, \ \text{"gato"}, \ \text{"come"}, \ \text{"pescado"}]$$

* Cuando el modelo evalúa la posición $t=0$ (`"El"`), solo puede ponderar información de `"El"`.
* Cuando evalúa $t=1$ (`"gato"`), puede atender a `["El", "gato"]`.
* Cuando evalúa $t=2$ (`"come"`), puede atender a `["El", "gato", "come"]`.
* Y así sucesivamente.

Para no procesar cada palabra una por una mediante bucles secuenciales lentos, **utilizamos matrices**. Esto nos permite calcular las relaciones de **todos los tokens en paralelo al mismo tiempo** en la GPU.

---

## 2. El Proceso en Tres Pasos

Para transformar puntuaciones de afinidad en probabilidades causales válidas, seguimos tres etapas matriciales:

### Paso 1: Matriz de Puntuaciones Brutas ($S$)
Calculamos las afinidades entre tokens (el producto $QK^T / \sqrt{d_k}$). Esta es una matriz cuadrada de tamaño $(T \times T)$:

$$S = \begin{bmatrix} s_{00} & s_{01} & s_{02} & s_{03} \\ s_{10} & s_{11} & s_{12} & s_{13} \\ s_{20} & s_{21} & s_{22} & s_{23} \\ s_{30} & s_{31} & s_{32} & s_{33} \end{bmatrix}$$

### Paso 2: Aplicación de la Máscara Triangular ($-\infty$)
Reemplazamos todas las posiciones por encima de la diagonal principal ($j > i$, el futuro) por $-\infty$:

$$S_{\text{masked}} = \begin{bmatrix} s_{00} & -\infty & -\infty & -\infty \\ s_{10} & s_{11} & -\infty & -\infty \\ s_{20} & s_{21} & s_{22} & -\infty \\ s_{30} & s_{31} & s_{32} & s_{33} \end{bmatrix}$$

### Paso 3: Transformación Softmax por Filas
Aplicamos Softmax a lo largo de la última dimensión (`dim=-1`). Dado que $e^{-\infty} = 0$, las posiciones futuras se anulan exactamente y cada fila suma $1.0$:

$$A = \text{Softmax}(S_{\text{masked}}) = \begin{bmatrix} 1.00 & 0.00 & 0.00 & 0.00 \\ a_{10} & a_{11} & 0.00 & 0.00 \\ a_{20} & a_{21} & a_{22} & 0.00 \\ a_{30} & a_{31} & a_{32} & a_{33} \end{bmatrix}$$

---

In [25]:
import torch
import torch.nn as nn
from torch.nn import functional as F

import numpy as np

In [26]:
# Creamos el tensor
thepast = torch.tensor([ 4,1,-2,-3 ])
N = len(thepast)

In [27]:
# Creamos los pesos (es lo mismo divir por la suma )
weights = torch.ones(N) / N

In [28]:
print(f'El pasado: {thepast}')
print(f'Pesos (importancia) del pasado: {weights}')
print(f'Suma de todos los pesos: {sum(weights)}')

El pasado: tensor([ 4,  1, -2, -3])
Pesos (importancia) del pasado: tensor([0.2500, 0.2500, 0.2500, 0.2500])
Suma de todos los pesos: 1.0


In [29]:
thepresent = sum(thepast * weights)
print(f'El presente (suma ponderada del pasado): {thepresent}')

El presente (suma ponderada del pasado): 0.0


# Integración Básica del Pasado: Ponderación Uniforme (Promedio Simple)

Antes de ver cómo el mecanismo de atención aprende qué partes del texto son más importantes, podemos entender la integración temporal mediante el caso más simple posible: **un promedio ponderado uniforme**.

---

## 1. Concepto y Código

Si representamos el historial pasado como un vector de activaciones, la forma más directa de resumir toda esa información en el momento presente es asignarle **exactamente el mismo peso a cada valor**:

```python
import torch

# 1. Definimos las activaciones del pasado (N = 4 puntos temporales)
thepast = torch.tensor([4.0, 1.0, -2.0, -3.0])
N = len(thepast)

# 2. Asignamos pesos uniformes normalizados (cada peso es 1/N)
weights = torch.ones(N) / N

print(f"El pasado: {thepast}")
print(f"Pesos (importancia) del pasado: {weights}")
print(f"Suma de todos los pesos: {weights.sum():.2f}")

# 3. Calculamos el presente como la suma ponderada del pasado
thepresent = (thepast * weights).sum()
print(f"El presente (suma ponderada del pasado): {thepresent:.2f}")

## Cambiando los pesos

##  El Problema de Pesos no Normalizados ($\sum > 1$)

Si los pesos de atención no están normalizados y su suma es mayor que $1$, nos enfrentamos a un problema crítico de **inestabilidad numérica**:

In [30]:
# Acá la suma de los pesos es mayor que uno
weights = torch.tensor([2, 1, 1, 1])
print(f'Suma de los pesos: {sum(weights)}. ¡Oh, oh...!')

Suma de los pesos: 5. ¡Oh, oh...!


#  Normalización Lineal vs. Softmax: ¿Por qué los LLMs usan Softmax?

Ambos métodos logran que los pesos sumen exactamente **$1.0$**, pero modifican la distribución de la atención de formas muy distintas:

In [31]:
# Podemos dividir por la suma de los pesos
linear_weights = weights / sum(weights)

# Podemos aplicar softmax
softmax_weights = torch.exp(weights) / sum(torch.exp(weights))

print(f'Pesos escalados: {linear_weights}')
print(f'\tSu suma: {sum(linear_weights)}')

print(f'\nPesos de softmax: {softmax_weights}')
print(f'\tSu suma: {sum(softmax_weights)}')

Pesos escalados: tensor([0.4000, 0.2000, 0.2000, 0.2000])
	Su suma: 1.0

Pesos de softmax: tensor([0.4754, 0.1749, 0.1749, 0.1749])
	Su suma: 1.0


In [32]:
thepresent_linear = sum(thepast * linear_weights)
thepresent_softmax = sum(thepast * softmax_weights)

print(f'El presente (suma lineal del pasado):  {thepresent_linear}')
print(f'El presente (suma softmax del pasado): {thepresent_softmax}')

El presente (suma lineal del pasado):  0.8000000715255737
El presente (suma softmax del pasado): 1.2019567489624023


In [33]:
# No estamos ignorando el futuro

# Impacto en la Representación del Presente

El método que elegimos para normalizar los pesos no es solo un detalle matemático: **cambia radicalmente el vector final del presente** que el modelo utilizará para predecir el futuro.

---

### Comparación de los Resultados

Usando los mismos valores pasados `thepast = [4, 1, -2, -3]`:

| Método | Ponderación | Valor del Presente | Comportamiento |
| :--- | :--- | :--- | :--- |
| **Suma Lineal** | Distribución plana / proporcional | `0.500` | Mezcla difusa de todo el contexto pasado. |
| **Softmax** | Distribución acentuada / focalizada | `1.025` | El modelo extrae con fuerza el valor más relevante (`4.0`) y atenúa el ruido negativo. |

---

### La Gran Conclusión sobre Attention

En el fondo, el mecanismo de atención de un LLM no es magia compleja:

1. Toma las activaciones de los tokens pasados.
2. Calcula qué tan relevante es cada uno usando una función de afinidad.
3. Aplica **Softmax** a ese vector para obtener una distribución de probabilidad que destaque lo importante y sume $1.0$.
4. Realiza una **suma ponderada** para sintetizar todo el pasado en una única representación para el presente.

## Ignorando el futuro

In [34]:
thedata = torch.tensor([4, 1, -2, -3, 8, 3, -1])
present_moment = 4
N = len(thedata)

print(f'Datos del pasado: {thedata[:present_moment]}')
print(f'El presente: {thedata[present_moment]}')
print(f'El futuro: {thedata[present_moment+1:]}')

Datos del pasado: tensor([ 4,  1, -2, -3])
El presente: 8
El futuro: tensor([ 3, -1])


#  Corte Temporal: Separando Pasado y Presente del Futuro

Para procesar secuencias y predecir el siguiente token de manera autorregresiva, debemos aislar exactamente qué porción de la información tiene permitida ver el modelo en cada paso de tiempo.

---

### Ejemplo con una Secuencia Temporal

Supongamos que tenemos un tensor con la siguiente secuencia de activaciones:

$$\text{Secuencia} = [4, \ 1, \ -2, \ -3, \ \mathbf{8}, \ 3, \ -1]$$

Si nos situamos en el **momento presente en el índice 4** (el valor $\mathbf{8}$):

* **Pasado y Presente (Información válida):**  
  Consideramos los elementos desde el índice $0$ hasta el $4$ $\rightarrow [4, \, 1, \, -2, \, -3, \, 8]$. El modelo integra todo este historial para calcular su estado actual.
* **Futuro (Información a ignorar):**  
  Los índices $5$ y $6$ $\rightarrow [3, \, -1]$ pertenecen al futuro. Deben enmascararse estrictamente (asignándoles $-\infty$) para que su peso en la Softmax sea **cero**.

---

> **Regla de causalidad en el índice $t$:**  
> Para cualquier posición temporal $t$, el modelo solo tiene acceso al subconjunto $\text{Secuencia}[0 : t+1]$, garantizando que la predicción dependa exclusivamente de lo que ya ha ocurrido.

In [35]:
# Creamos un vector de pesos que ponemos todos en 1
past_weights = torch.ones(N)

# El presente mas un paso de tiempo sera igual a 0
past_weights[present_moment+1:] = 0
past_weights


tensor([1., 1., 1., 1., 1., 0., 0.])

Asi que esta es una ponderacion sin escala asi que la suma seguira siendo mayor que 1

In [36]:
# Vamos a dividir por la suma del pasado en este vector
past_weights_linear = past_weights / torch.sum(past_weights)

print(f'Pesos escalados: {past_weights_linear}')
print(f'\tSu suma: {sum(past_weights_linear)}')

Pesos escalados: tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000])
	Su suma: 1.0


In [37]:
# Aplicar softmax a los pesos con ceros
past_weights_softmax = torch.exp(past_weights) / torch.sum(torch.exp(past_weights))

print(f'Pesos de softmax: {past_weights_softmax}')
print(f'\tSu suma: {sum(past_weights_softmax)}')

Pesos de softmax: tensor([0.1743, 0.1743, 0.1743, 0.1743, 0.1743, 0.0641, 0.0641])
	Su suma: 1.0


In [38]:
#Al convertirlos en una funcion de probabilidad los valores no son totalmente ceros (porque el cero es 1)

In [39]:
# Por ejemplo:
torch.exp(torch.tensor([0]))

tensor([1.])

In [40]:
# asi que el valor de -10 sigue siendo un valor distinto a cero pero como sabemos si se va haciendo mas pequeño nos dara 0
torch.exp(torch.tensor([-10]))

tensor([4.5400e-05])

In [41]:
# Tecnicamente no es cero pero pytorch le da este valor 0
torch.exp(torch.tensor([-10000]))

tensor([0.])

## Entonces que hacemos? la formula de llevar los valores al inf

In [42]:
# Recrear los pesos para el pasado, pero estableciendo los valores futuros en -infinito
past_weights = torch.ones(N)
past_weights[present_moment + 1 :] = -torch.inf

# Aplicar softmax
past_weights_softmax = torch.exp(past_weights) / torch.sum(
    torch.exp(past_weights)
)

# Imprimir los resultados
print(f'Pesos sin escalar: {past_weights}')
print(f'Pesos escalados: {past_weights_softmax}')
print(f'\tSu suma: {sum(past_weights_softmax)}')

Pesos sin escalar: tensor([1., 1., 1., 1., 1., -inf, -inf])
Pesos escalados: tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000])
	Su suma: 1.0


Como podemos observar 1s para el presente y pasado -inf para el futuro y la ecuacion nos dice que el limite de e a la x cuando tiende a -inf es = a 0 y la suma total de las probabilidades de softmax ya nos da 1

## Escalar esto para una matriz

In [43]:
trill = torch.tril(torch.ones(9,9))

In [44]:
trill

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [45]:
trill[trill==0] = -torch.inf
trill

tensor([[1., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [47]:
trill_softmax = F.softmax(trill,dim=-1)
trill_softmax

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.0000],
        [0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111]])

In [49]:
for timepoint in range(trill.shape[0]):
  print(f'\nPesos para el cálculo en el punto temporal {timepoint}:')
  print(f'\t{trill_softmax[timepoint]}')


Pesos para el cálculo en el punto temporal 0:
	tensor([1., 0., 0., 0., 0., 0., 0., 0., 0.])

Pesos para el cálculo en el punto temporal 1:
	tensor([0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

Pesos para el cálculo en el punto temporal 2:
	tensor([0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

Pesos para el cálculo en el punto temporal 3:
	tensor([0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

Pesos para el cálculo en el punto temporal 4:
	tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000, 0.0000])

Pesos para el cálculo en el punto temporal 5:
	tensor([0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000, 0.0000])

Pesos para el cálculo en el punto temporal 6:
	tensor([0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000, 0.0000])

Pesos para el cálculo en el punto temporal 7:
	tensor([0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.0000])

Pes

In [ ]:
# por que no tebenis unos cuando aplicamos sofrmax?

# ¿Por qué no vemos ceros en el futuro? El error de enmascarar con `0` en lugar de `-\infty`

Si al aplicar Softmax a una matriz triangular obtienes valores como `0.2536` y `0.0933` en lugar de `1.0` y `0.0`, significa que **el futuro fue rellenado con `0` en vez de `-\infty`**.

---

## 1. El Origen del Problema: $e^0 = 1$

Analicemos qué pasó en la **primera fila** de la matriz de entrada:

$$\text{Fila 0 de entrada} = [1, \, 0, \, 0, \, 0, \, 0, \, 0, \, 0, \, 0, \, 0]$$

Se colocó un $1$ en el presente y $0$ en las $8$ posiciones futuras esperando que el $0$ las anulara. Sin embargo, la función **Softmax eleva todo a la exponencial ($e^x$)**:

1. **Exponencial de los valores:**
   * Para el presente ($1$): $e^1 \approx 2.7183$
   * Para cada posición del futuro ($0$): $e^0 = \mathbf{1.0}$ *(¡el cero no se anula, se convierte en 1!)*

2. **Suma de la fila (denominador de Softmax):**
   $$\sum = e^1 + 8 \cdot (e^0) = 2.7183 + 8 \cdot (1.0) = \mathbf{10.7183}$$

3. **Valores resultantes tras Softmax:**
   * **Presente:** $\frac{e^1}{10.7183} = \frac{2.7183}{10.7183} = \mathbf{0.2536}$
   * **Cada posición futura:** $\frac{e^0}{10.7183} = \frac{1.0}{10.7183} = \mathbf{0.0933}$

---

## 2. Las Dos Consecuencias Críticas

* **El futuro no se ignora:** Cada uno de los 8 tokens del futuro se queda con un **$9.33\%$** de atención (sumando casi el $75\%$ de la probabilidad total).
* **El presente se diluye:** En lugar de concentrar el **$100\%$ ($1.0$)** de la atención en el único token conocido, solo retiene el **$25.36\%$**.

---

## 3. La Solución: Enmascarar con `float('-inf')`

Para que el resultado de Softmax sea verdaderamente $0$, la entrada en las posiciones futuras debe ser **$-\infty$**:

$$\lim_{x \to -\infty} e^x = 0$$


In [ ]:
# Definimos una secuencia de tamaño T = 5 (o 9)
T = 5

# 1. Matriz de ceros como logits base (todas las posiciones pasadas pesan igual inicialmente)
scores = torch.zeros(T, T)

# 2. Máscara triangular inferior (1 en pasado/presente, 0 en futuro)
mask = torch.tril(torch.ones(T, T))

# 3. Llenamos el futuro (donde mask == 0) con -inf
causal_scores = scores.masked_fill(mask == 0, float("-inf"))

# 4. Aplicamos Softmax por filas (última dimensión)
causal_weights = F.softmax(causal_scores, dim=-1)

# Mostramos el resultado formateado
print("Matriz de Pesos de Atención Causal:")
print(torch.round(causal_weights, decimals=4))

Final con activaciones random

In [50]:
activations = torch.randn(N, N)
tril = torch.tril(torch.ones(N, N))

print('-- ACTIVACIONES ORIGINALES:')
print(activations)

print('\n-- FACTOR DE PONDERACIÓN DEL PASADO:')
print(tril)

scaled_activations = activations * tril
scaled_activations[scaled_activations == 0] = -torch.inf
print('\n-- ACTIVACIONES ESCALADAS DEL PASADO:')
print(scaled_activations)

softmax_past = F.softmax(scaled_activations, dim=-1)
print('\n-- ACTIVACIONES DEL PASADO TRAS SOFTMAX:')
print(softmax_past)

-- ACTIVACIONES ORIGINALES:
tensor([[ 1.2971,  1.4401, -0.0096,  1.6891, -0.1528, -1.2286, -1.4205],
        [ 1.3086,  0.5050,  0.3041,  1.8283, -1.1847,  1.1748, -0.4485],
        [-0.2473,  1.5068,  1.5992, -0.8971, -0.4140,  0.6678,  1.2761],
        [ 0.1748,  1.2428,  0.1389,  1.7178, -0.2355,  0.2488,  1.7138],
        [-0.8881, -0.1032, -0.9928,  0.4926, -0.9010,  0.6279, -0.6665],
        [-1.4110,  0.0580, -0.1022,  1.0021,  0.6347,  0.1854, -0.5559],
        [ 0.3588,  0.2137,  1.4902, -0.2111, -0.3652, -1.3104,  1.1884]])

-- FACTOR DE PONDERACIÓN DEL PASADO:
tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])

-- ACTIVACIONES ESCALADAS DEL PASADO:
tensor([[ 1.2971,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.3086,  0.5050,    -inf,    -inf

In [51]:
# Confirmar:
torch.sum(softmax_past, dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

## Para que lo tengas en cuenta algunas alternativas

In [52]:
import time

nIters = int(2e5)

# Opción 1: encontrar ceros y -torch.inf
start_time = time.time()
for _ in range(nIters):
  tril = torch.tril(torch.ones(10, 10))
  tril[tril == 0] = -torch.inf
print(f'Opción 1: {time.time()-start_time:.3f} s')

# Opción 2: encontrar ceros y float('-inf')
start_time = time.time()
for _ in range(nIters):
  tril = torch.tril(torch.ones(10, 10))
  tril[tril == 0] = float('-inf')
print(f'Opción 2: {time.time()-start_time:.3f} s')

# Opción 3: masked_fill con float('-inf')
start_time = time.time()
for _ in range(nIters):
  tril = torch.tril(torch.ones(10, 10))
  tril = tril.masked_fill(tril == 0, float('-inf'))
print(f'Opción 3: {time.time()-start_time:.3f} s')

Opción 1: 5.425 s
Opción 2: 4.201 s
Opción 3: 4.271 s
